In [1]:
!pip install -q ultralytics opencv-python numpy

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import cv2
import numpy as np
import collections
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment

# Константы и индексы точек в разметке COCO (-шнене)
L_SHOULDER, R_SHOULDER = 5, 6
L_ELBOW, R_ELBOW = 7, 8
L_WRIST, R_WRIST = 9, 10
L_HIP, R_HIP = 11, 12
L_KNEE, R_KNEE = 13, 14

SKELETON_PAIRS = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)
]

# Настройки цвета
LOWER_RED1, UPPER_RED1 = np.array([0, 100, 70]), np.array([12, 255, 255])
LOWER_RED2, UPPER_RED2 = np.array([165, 100, 70]), np.array([180, 255, 255])
LOWER_BLUE, UPPER_BLUE = np.array([90, 100, 70]), np.array([130, 255, 255])
LOWER_WHITE, UPPER_WHITE = np.array([0, 0, 150]), np.array([180, 60, 255])

COLOR_RED_BGR = (0, 0, 255)
COLOR_BLUE_BGR = (255, 50, 0)
COLOR_REF_BGR = (200, 200, 200)

# Функции анализа цвета
def get_polygon_crop_and_mask(frame, target_kps):
    """Строит полигон, вырезает фрагмент (crop) и возвращает маску для максимальной скорости"""
    frame_h, frame_w = frame.shape[:2]
    if any(kp[2] < 0.4 for kp in target_kps):
        return None, None

    pts = np.array([[kp[0], kp[1]] for kp in target_kps], dtype=np.int32)
    x, y, w, h = cv2.boundingRect(pts)

    x1, y1 = max(0, x), max(0, y)
    x2, y2 = min(frame_w, x + w), min(frame_h, y + h)

    if x2 <= x1 or y2 <= y1:
        return None, None

    crop = frame[y1:y2, x1:x2]
    mask = np.zeros((crop.shape[0], crop.shape[1]), dtype=np.uint8)
    pts_shifted = pts - [x1, y1]
    cv2.fillPoly(mask, [pts_shifted], 255)

    return crop, mask

def get_glove_crop_and_mask(frame, elbow_kp, wrist_kp):
    """Вычисляет позицию перчатки, вырезает маленький квадрат и возвращает круглую маску"""
    if elbow_kp[2] < 0.4 or wrist_kp[2] < 0.4:
        return None, None

    ex, ey = elbow_kp[:2]
    wx, wy = wrist_kp[:2]

    # Продлеваем вектор от локтя к запястью на 40%
    vx, vy = wx - ex, wy - ey
    gx, gy = int(wx + vx * 0.4), int(wy + vy * 0.4)

    frame_h, frame_w = frame.shape[:2]
    x1, y1 = max(0, gx - 15), max(0, gy - 15)
    x2, y2 = min(frame_w, gx + 15), min(frame_h, gy + 15)

    if x2 <= x1 or y2 <= y1:
        return None, None

    crop = frame[y1:y2, x1:x2]
    mask = np.zeros((crop.shape[0], crop.shape[1]), dtype=np.uint8)

    # Рисуем круг на маске относительно вырезанного куска
    cx, cy = gx - x1, gy - y1
    cv2.circle(mask, (cx, cy), 15, 255, -1)

    return crop, mask

def extract_colors(crop, mask):
    """Подсчет пикселей внутри вырезанного фрагмента"""
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    roi_pixels = cv2.bitwise_and(hsv, hsv, mask=mask)

    total_pixels = np.count_nonzero(mask)
    if total_pixels == 0:
        return 0.0, 0.0, 0.0

    mask_red = cv2.bitwise_or(cv2.inRange(roi_pixels, LOWER_RED1, UPPER_RED1),
                              cv2.inRange(roi_pixels, LOWER_RED2, UPPER_RED2))
    mask_blue = cv2.inRange(roi_pixels, LOWER_BLUE, UPPER_BLUE)
    mask_white = cv2.inRange(roi_pixels, LOWER_WHITE, UPPER_WHITE)

    return (np.sum(mask_red > 0) / total_pixels,
            np.sum(mask_blue > 0) / total_pixels,
            np.sum(mask_white > 0) / total_pixels)

def classify_person_features(frame, kps):
    """Собирает цвет с торса, шорт и перчаток, взвешивая их через нелинейную уверенность"""
    total_red, total_blue, total_ref = 0.0, 0.0, 0.0
    weight_sum = 0.0001 # Защита от деления на 0

    # Торс
    torso_kps = [kps[L_SHOULDER], kps[R_SHOULDER], kps[R_HIP], kps[L_HIP]]
    torso_crop, torso_mask = get_polygon_crop_and_mask(frame, torso_kps)
    if torso_crop is not None:
        # Нелинейная уверенность (в кубе)
        conf_w = (sum(kp[2] for kp in torso_kps) / 4) ** 3
        r, b, w = extract_colors(torso_crop, torso_mask)
        total_red += r * conf_w
        total_blue += b * conf_w
        total_ref += w * conf_w
        weight_sum += conf_w

    # Шорты (от бедра до колена)
    shorts_kps = [kps[L_HIP], kps[R_HIP], kps[R_KNEE], kps[L_KNEE]]
    shorts_crop, shorts_mask = get_polygon_crop_and_mask(frame, shorts_kps)
    if shorts_crop is not None:
        conf_w = (sum(kp[2] for kp in shorts_kps) / 4) ** 3
        r, b, _ = extract_colors(shorts_crop, shorts_mask)
        total_red += r * conf_w
        total_blue += b * conf_w
        weight_sum += conf_w

    # Перчатки
    for elbow_idx, wrist_idx in [(L_ELBOW, L_WRIST), (R_ELBOW, R_WRIST)]:
        g_crop, g_mask = get_glove_crop_and_mask(frame, kps[elbow_idx], kps[wrist_idx])
        if g_crop is not None:
            # У перчаток берем conf_w и чуть занижаем вес (0.7), так как они маленькие
            conf_w = ((kps[elbow_idx][2] + kps[wrist_idx][2]) / 2) ** 3 * 0.7
            r, b, _ = extract_colors(g_crop, g_mask)
            total_red += r * conf_w
            total_blue += b * conf_w
            weight_sum += conf_w

    return total_red / weight_sum, total_blue / weight_sum, total_ref / weight_sum

# Менеджер ролей через венгерский алгоритм
class BoxingRoleManager:
    def __init__(self):
        # Память о координатах бойцов на предыдущем кадре
        self.prev_centers = {'RED': None, 'BLUE': None}

    def update(self, frame, track_ids, bboxes, keypoints):
        frame_h, frame_w = frame.shape[:2]
        frame_diag = np.sqrt(frame_w**2 + frame_h**2)

        candidates = []

        # Извлекаем признаки всех людей на кадре
        for tid, bbox, kps in zip(track_ids, bboxes, keypoints):
            if (bbox[3] - bbox[1]) < frame_h * 0.2:
                continue

            red, blue, ref = classify_person_features(frame, kps)

            # Отсекаем явного рефери (много белого, мало красного/синего)
            if ref > 0.15 and red < 0.1 and blue < 0.1:
                continue

            cx, cy = (bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2
            candidates.append({
                'tid': tid, 'bbox': bbox, 'kps': kps, 'center': (cx, cy),
                'score_red': red, 'score_blue': blue
            })

        output_fighters = {}
        if not candidates:
            return output_fighters

        # Строим матрицу стоимости (Кандидаты x Роли: Красный, Синий)
        cost_matrix = np.zeros((len(candidates), 2))

        for i, cand in enumerate(candidates):
            # Считаем пространственную память: если человек стоит там же, где был Синий, добавляем ему балл
            s_red = cand['score_red']
            s_blue = cand['score_blue']

            if self.prev_centers['RED']:
                dist = np.linalg.norm(np.array(cand['center']) - np.array(self.prev_centers['RED']))
                # 0.3 диагонали - максимальное смещение, дальше память не работает
                mem_score = max(0.0, 1.0 - (dist / (frame_diag * 0.3)))
                s_red = s_red * 0.6 + mem_score * 0.4

            if self.prev_centers['BLUE']:
                dist = np.linalg.norm(np.array(cand['center']) - np.array(self.prev_centers['BLUE']))
                mem_score = max(0.0, 1.0 - (dist / (frame_diag * 0.3)))
                s_blue = s_blue * 0.6 + mem_score * 0.4

            # Заполняем матрицу (Венгерский алгоритм минимизирует значения, поэтому 1.0 - score)
            cost_matrix[i, 0] = 1.0 - s_red
            cost_matrix[i, 1] = 1.0 - s_blue

        # Находим оптимальное назначение (без жесткой привязки к track_id)
        row_ind, col_ind = linear_sum_assignment(cost_matrix)

        for r, c in zip(row_ind, col_ind):
            role = 'RED' if c == 0 else 'BLUE'
            # Защита: назначаем только если итоговый скор (цвет + позиция) адекватный (ошибка < 0.85)
            if cost_matrix[r, c] < 0.85:
                output_fighters[role] = candidates[r]
                self.prev_centers[role] = candidates[r]['center'] # Запоминаем центр для след. кадра

        # Очищаем память потерянных ролей
        if 'RED' not in output_fighters: self.prev_centers['RED'] = None
        if 'BLUE' not in output_fighters: self.prev_centers['BLUE'] = None

        return output_fighters

# Отрисовка и главный процесс
def draw_fighter(frame, fighter_data, color_bgr, label):
    bbox, kps = fighter_data['bbox'], fighter_data['kps']
    x1, y1, x2, y2 = map(int, bbox)

    cv2.rectangle(frame, (x1, y1), (x2, y2), color_bgr, 2)
    cv2.rectangle(frame, (x1, y1 - 25), (x1 + 180, y1), color_bgr, -1)
    cv2.putText(frame, label, (x1 + 5, y1 - 7), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    for start_idx, end_idx in SKELETON_PAIRS:
        if start_idx < len(kps) and end_idx < len(kps):
            kp1, kp2 = kps[start_idx], kps[end_idx]
            if kp1[2] > 0.4 and kp2[2] > 0.4:
                cv2.line(frame, (int(kp1[0]), int(kp1[1])), (int(kp2[0]), int(kp2[1])), color_bgr, 3)

    for kp in kps:
        if kp[2] > 0.4:
            cv2.circle(frame, (int(kp[0]), int(kp[1])), 4, (255, 255, 255), -1)

def process_boxing_video(input_path, output_path):
    print("Загрузка YOLOv8 Pose...")
    model = YOLO('yolo26m-pose.pt')

    cap = cv2.VideoCapture(input_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    manager = BoxingRoleManager()
    frame_count = 0

    print("Анализ видео запущен...")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        results = model.track(frame, persist=True, classes=[0], conf=0.35, verbose=False)

        if results[0].boxes is not None and results[0].boxes.id is not None:
            bboxes = results[0].boxes.xyxy.cpu().numpy()
            track_ids = results[0].boxes.id.int().cpu().numpy()
            keypoints = results[0].keypoints.data.cpu().numpy()

            fighters = manager.update(frame, track_ids, bboxes, keypoints)

            if 'RED' in fighters:
                draw_fighter(frame, fighters['RED'], COLOR_RED_BGR, f"RED ID:{fighters['RED']['tid']}")
            if 'BLUE' in fighters:
                draw_fighter(frame, fighters['BLUE'], COLOR_BLUE_BGR, f"BLUE ID:{fighters['BLUE']['tid']}")

        out.write(frame)
        if frame_count % 200 == 0:
            print(f"Обработано кадров: {frame_count}")

    cap.release()
    out.release()
    print(f"Готово! Результат сохранен в {output_path}")

INPUT_VIDEO = '/content/drive/MyDrive/boxing/new_boxing_tournament/Раунд1.mp4'
OUTPUT_VIDEO = '/content/output_tracked.mp4'

process_boxing_video(INPUT_VIDEO, OUTPUT_VIDEO)

Загрузка YOLOv8 Pose...
Анализ видео запущен...
Обработано кадров: 30
Обработано кадров: 60
Обработано кадров: 90
Обработано кадров: 120
Обработано кадров: 150
Обработано кадров: 180
Обработано кадров: 210
Обработано кадров: 240
Обработано кадров: 270
Обработано кадров: 300
Обработано кадров: 330
Обработано кадров: 360
Обработано кадров: 390
Обработано кадров: 420
Обработано кадров: 450
Обработано кадров: 480
Обработано кадров: 510
Обработано кадров: 540
Обработано кадров: 570
Обработано кадров: 600
Обработано кадров: 630
Обработано кадров: 660
Обработано кадров: 690
Обработано кадров: 720
Обработано кадров: 750
Обработано кадров: 780
Обработано кадров: 810
Обработано кадров: 840
Обработано кадров: 870
Обработано кадров: 900
Обработано кадров: 930
Обработано кадров: 960
Обработано кадров: 990
Обработано кадров: 1020
Обработано кадров: 1050
Обработано кадров: 1080
Обработано кадров: 1110
Обработано кадров: 1140
Обработано кадров: 1170
Обработано кадров: 1200
Обработано кадров: 1230
Обра